# 08. Introduction to the Keras Tuner

### Overview

The Keras tuner is a library that helps you pick the optimal set of hyperparameters for your TensorFlow program. The process of selecting the right set of hyperparameters for your machine learning (ML) application is called _hyperparameter tuning_ or _hypertuning_.

Hyperparameters are the variables that govern the training process and the topology of an ML model. These variables remain constant over the training process and directly impact the performance of your ML program. Hyperparameters are of two types:

1. **Model hyperparameters** which influence model selection such as the number and width of hidden layers

2. **Algorithm hyperparameters** which influence the speed and quality of the learning algorithm such as the learning rate for Stochastic Gradient Descent (SGD) and the number of nearest neighbors for a k Nearest Neighbors (KNN) classifier

In this tutorial, you will use the Keras Tuner to perform hypertuning for an image classification application.

### Setup

In [1]:
import tensorflow as tf
from tensorflow import keras

Install and import the Keras Tuner.

In [2]:
# pip install -q -U keras-tuner

In [4]:
import keras_tuner as kt

### Download and prepare the dataset

In this tutorial, you will use the Keras Tuner to find the best hyperparameters for a machine learning model that classifies images of clothing from the Fashion MNIST dataset.

Load the data.

In [5]:
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()

In [6]:
# Normalize pixel values between 0 and 1
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

In [9]:
# img_train[0]

In [10]:
label_train

array([9, 0, 0, ..., 3, 0, 5], shape=(60000,), dtype=uint8)

In [12]:
# img_test[0]

In [13]:
label_test

array([9, 2, 1, ..., 8, 1, 5], shape=(10000,), dtype=uint8)

### Define the model

When you build a model for hypertuning, you also define the hyperparameter search space in addition to the model architecture. The model you set up for hypertuning is called a _hypermodel_.

You can define a hypermodel through two approaches:

* By using a model builder function
* By subclassing the `HyperModel` class of the Keras Tuner API

You can also use two pre-defined HyperModel classes - HyperXception and HyperResNet for computer vision applications.

In this tutorial, you use a model builder function to define the image classification model. The model builder function returns a compiled model and uses hyperparameters you define inline to hypertune the model.

In [14]:
def model_builder(hp):
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=(28, 28)))

    # Tune the number of units in the first Dense layer
    # Choose an optimal value between 32-512
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(keras.layers.Dense(units=hp_units, activation='relu'))
    model.add(keras.layers.Dense(10))

    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001 or 0.0001
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])

    return model

### Instantiate the tuner and perform hypertuning

Instantiate the tuner to perform the hypertuning. The Keras Tuner has four tuners available, `RandomSearch`, `Hyperband`, `BayesianOptimization` and `Sklearn`. In this tutorial, you use the Hyperband tuner.

To instantiate the Hyperband tuner, you must specify the hypermodel, the `objective` to optimize and the maximum number of epochs to train (`max_epochs`).

In [15]:
tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='my_dir',
                     project_name='intro_to_kt')

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


The Hyperband tuning algorithm uses adaptive resource allocation and early-stopping to quickly converge on a high-performing model. This is done using a sports championship style bracket. The algorithm trains a large number of models for a few epochs and carries forward only the top-performing half of models to the next round. Hyperband determines the number of models to train in a bracket by computing 1 + log _factor (`max_epochs`) and rounding it up to the nearest integer.

Create a callback to stop training early after reaching a certain value for the validation loss.

In [16]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

Run the hyperparameter search. The arguments for the search method are the same as those used for `tf.keras.model.fit` in addition to the callback above.

In [18]:
tuner.search(img_train, label_train, epochs=50, validation_split=0.2, callbacks=[stop_early])

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is {best_hps.get('units')} and the optimal learning rate for the optimizer is
{best_hps.get('learning_rate')}.
""")

Trial 30 Complete [00h 04m 07s]
val_accuracy: 0.8910833597183228

Best val_accuracy So Far: 0.8920833468437195
Total elapsed time: 00h 46m 36s

The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is 256 and the optimal learning rate for the optimizer is
0.001.



### Train the model

Find the optimal number of epochs to train the model with the hyperparameters obtained from the search.

In [21]:
# Build the model with the optimal hyperparameters and train it on the data for 25 epochs
model = tuner.hypermodel.build(best_hps)
history = model.fit(img_train, label_train, epochs=25, validation_split=0.2)

Epoch 1/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - accuracy: 0.8242 - loss: 0.5012 - val_accuracy: 0.8443 - val_loss: 0.4222
Epoch 2/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - accuracy: 0.8639 - loss: 0.3746 - val_accuracy: 0.8625 - val_loss: 0.3855
Epoch 3/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.8756 - loss: 0.3376 - val_accuracy: 0.8788 - val_loss: 0.3375
Epoch 4/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.8844 - loss: 0.3108 - val_accuracy: 0.8732 - val_loss: 0.3520
Epoch 5/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.8919 - loss: 0.2892 - val_accuracy: 0.8792 - val_loss: 0.3431
Epoch 6/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.8961 - loss: 0.2743 - val_accuracy: 0.8832 - val_loss: 0.3212
Epoch 7/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - accuracy: 0.9017 - loss: 0.2605 - val_accuracy: 0.8863 - val_loss: 0.3209
Epoch 8/25
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 13ms/step - accuracy: 0.9060 -

In [22]:
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print("Best epoch: %d" % (best_epoch,))

Best epoch: 23


Re-instantiate the hypermodel and train it with the optimal number of epochs from above.

In [23]:
hypermodel = tuner.hypermodel.build(best_hps)

# Retrain the model
hypermodel.fit(img_train, label_train, epochs=best_epoch, validation_split=0.2)

Epoch 1/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - accuracy: 0.8242 - loss: 0.5000 - val_accuracy: 0.8502 - val_loss: 0.4145
Epoch 2/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - accuracy: 0.8650 - loss: 0.3734 - val_accuracy: 0.8629 - val_loss: 0.3788
Epoch 3/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - accuracy: 0.8771 - loss: 0.3322 - val_accuracy: 0.8765 - val_loss: 0.3457
Epoch 4/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.8864 - loss: 0.3086 - val_accuracy: 0.8791 - val_loss: 0.3393
Epoch 5/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.8929 - loss: 0.2895 - val_accuracy: 0.8836 - val_loss: 0.3281
Epoch 6/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.8998 - loss: 0.2744 - val_accuracy: 0.8851 - val_loss: 0.3153
Epoch 7/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.9026 - loss: 0.2623 - val_accuracy: 0.8817 - val_loss: 0.3316
Epoch 8/23
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.9055 -

To finish this tutorial, evaluate the hypermodel on the test data.

In [24]:
eval_result = hypermodel.evaluate(img_test, label_test)
print("[test loss, test accuracy]:", eval_result)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8821 - loss: 0.4093
[test loss, test accuracy]: [0.40927159786224365, 0.882099986076355]


The `my_dir/intro_to_kt` directory contains detailed logs and checkpoints for every trial (model configuration) run during the hyperparameter search. If you re-run the hyperparameter search, the Keras Tuner uses the existing state from these logs to resume the search. To disable this behavior, pass an additional `overwrite=True` argument while instantiating the tuner.

### Summary

In this tutorial, you learned how to use the Keras Tuner to tune hyperparameters for a model. To learn more about the Keras Tuner, check out these additional resources:

* Keras Tuner on the TensorFlow blog
* Keras Tuner website

Also check out the HParams Dashboard in TensorBoard to interactively tune your model hyperparameters.